# nllb-tani-1.3B — fine-tuning NLLB-200 (distilled 1.3B) on the Tani corpus, same methodology as m2mTani-base

Fine-tunes **`facebook/nllb-200-distilled-1.3B`** for **9 languages in all 72 directions** — Adi `adi`, Apatani `apt`,
Galo `adl`, Mising `mrg`, Nyishi `njz`, Tagin `tgj`, Hruso `hru`, English `eng`, Hindi `hin` — with the data, split,
training recipe and evaluation of the from-scratch **m2mTani-base** (notebook `02m2mtani_base_multilingual.ipynb`), so
the two models can be compared number for number.

**Identical to m2mTani-base**

| part | what is reused |
|---|---|
| data | `tani_corpus.json` (SHA-256 checked) |
| split | the **exact** `split_manifest.json` of the base run (whole chapters, global across languages, near-copies moved to train); it is loaded, verified and **never rebuilt** |
| training stream | 35 % dialect↔English · 35 % dialect↔Hindi · 20 % dialect↔dialect · 10 % English↔Hindi; 10 % denoising (15 % of words masked/deleted); temperature sampling T = 5; 3 English versions per entry per epoch; modern-English targets; dictionary weight 0.3; source subword sampling α = 0.1; max 256 pieces, length ratio ≤ 3; length-bucketed batches; 24,000 target tokens per update |
| optimisation | AdamW (β 0.9/0.98, ε 1e-8, weight decay 1e-4 except norms/biases), inverse-square-root schedule, gradient clipping 1.0, label smoothing 0.1, **R-Drop α 5**, bf16 |
| selection | dev chrF++ (greedy, 40 sentences per direction) every 2,000 updates; early stop after 5 evaluations without a 0.1 gain; top-5 checkpoints kept; average vs best single, decided by beam-5 dev chrF++ |
| evaluation | the same test items (`test_per_direction` 500, same seeds), chrF++ / multi-reference BLEU / TER / CER / length ratio, copy-the-source baseline, copy / off-target / empty / hallucination diagnostics; **plus** the leak-free Test-data set (109 test + 132 dev sentences) with bootstrap 95 % intervals |

**What changes, and why (forced by the pretrained model)**

| m2mTani-base | here | why |
|---|---|---|
| own SentencePiece, 24k pieces, trained on the train split | NLLB's SentencePiece (256k pieces), unchanged | the pretrained embeddings belong to that vocabulary |
| tags `<2tgj>` … | NLLB codes `eng_Latn`, `hin_Deva` (existing) + **7 new codes** `adi_Latn`, `apt_Latn`, `adl_Latn`, `mrg_Latn`, `njz_Latn`, `tgj_Latn`, `hru_Latn` | NLLB-200 has none of the Tani-region languages; new code embeddings start from the mean of `lus_Latn` (Mizo) and `kac_Latn` (Jingpho), Tibeto-Burman languages in Latin script, plus small noise |
| **target** tag on the source, forced as first output token | NLLB convention: **source** code first on the encoder side, **target** code forced as the first decoder token | the way NLLB was pretrained |
| random init, LR 5e-4, 4,000 warm-up | pretrained, **LR 1e-4, 1,000 warm-up** (same schedule shape) | a from-scratch LR would destroy the pretrained weights |
| dropout 0.2 | NLLB's own dropout (0.1) | keep the pretrained regime |
| 12,000-token micro-batches | 800–2,400 (by GPU memory), more accumulation to the same 24k tokens/update; gradient checkpointing | 1.37B parameters and a 256k output layer (×2 with R-Drop) |
| fp32 checkpoints | top-k checkpoints stored in bf16 (averaged in fp32) | 2.7 GB instead of 5.5 GB each |

**Tasks:** T1 corpus · T2 split (verify exact partition) · T3 tokenizer + new language codes · T4 stream preview ·
T5 model · T6 fine-tuning · T7 test-chapter evaluation + comparison with m2mTani-base · T8 leak-free Test-data
evaluation + comparison · T9 translation demo. Every task loads what it needs; training resumes after a disconnect.

**Hardware:** one CUDA GPU with ≥ 40 GB (A100 40/80 GB tested in design; 80 GB recommended for R-Drop). CPU/MPS only for
smoke tests (`MTANI_DEBUG_BOOKS=JHN,PHM` and `MTANI_MAX_UPDATES=20`).

**Licence:** NLLB-200 weights are **CC BY-NC 4.0** — the fine-tuned model is for non-commercial use.

## Setup 1 · Device check (run first)

Lists the accelerators and their memory. Fine-tuning a 1.37B-parameter model needs a CUDA GPU with at least 40 GB;
Apple GPUs (MPS) and the CPU can run the data and evaluation tasks and tiny smoke tests only. Force a device with
`MTANI_DEVICE=cuda|mps|cpu` (default `auto`: CUDA → MPS → CPU).

In [ ]:
import os, platform
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # before CUDA starts
import torch

DEVICE_CHOICE = os.environ.get('MTANI_DEVICE', 'auto').lower()
print(f'torch {torch.__version__} | {platform.platform()} | {os.cpu_count()} CPU cores')
GPU_GIB = 0.0
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'CUDA GPU {i}: {p.name} | {p.total_memory / 2**30:.1f} GiB | bf16 {torch.cuda.is_bf16_supported()}')
    GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
else:
    print('CUDA GPU: not available')
print('Apple GPU (MPS):', 'available' if torch.backends.mps.is_available() else 'not available')
AVAILABLE = {'cuda': torch.cuda.is_available(), 'mps': torch.backends.mps.is_available(), 'cpu': True}
picked = next(d for d in ('cuda', 'mps', 'cpu') if AVAILABLE[d]) if DEVICE_CHOICE == 'auto' else DEVICE_CHOICE
assert picked in AVAILABLE and AVAILABLE[picked], f'{picked} is not available on this machine'
x = torch.randn(256, 256, device=picked); (x @ x).sum().item()           # smoke test on the chosen device
print(f'-> {picked}')
if picked != 'cuda':
    print('WARNING: no CUDA GPU. Fine-tuning NLLB-1.3B needs >= 40 GB of GPU memory; use this device for the data, '
          'tokenizer and evaluation tasks or tiny smoke tests only.')
elif GPU_GIB < 38:
    print(f'WARNING: {GPU_GIB:.0f} GiB GPU. Training will use very small micro-batches; consider MTANI_RDROP_ALPHA=0 '
          '(halves memory, deviates from the base recipe).')

## Setup 2 · Runtime (Google Colab: A100 GPU + High-RAM)

On Colab this mounts Google Drive and keeps everything that must survive a disconnect there
(`MyDrive/mtani/nllb-tani-1.3B-finetune/runs` and `/checkpoints`). Upload the `m2m-tani-base-57M` folder to
`MyDrive/mtani/` first: the notebook reads the corpus, the split manifest and the leak-free test set from it. Outside
Colab this cell does nothing.

In [ ]:
import os, sys, shutil, subprocess, importlib.util
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = os.environ.get('MTANI_DRIVE_DIR', '/content/drive/MyDrive/mtani')
    need = [m for m in ('sacrebleu', 'sentencepiece') if importlib.util.find_spec(m) is None]
    if need:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
    os.environ.setdefault('MTANI_BUNDLE_DIR', f'{DRIVE_DIR}/m2m-tani-base-57M')
    os.environ.setdefault('MTANI_RUN_DIR', f'{DRIVE_DIR}/nllb-tani-1.3B-finetune/runs')
    os.environ.setdefault('MTANI_CKPT_DIR', f'{DRIVE_DIR}/nllb-tani-1.3B-finetune/checkpoints')
    os.environ.setdefault('MTANI_JOB_HOURS', '23')
    src = os.path.join(os.environ['MTANI_BUNDLE_DIR'], '02_data', 'tani_corpus.json')
    dst = '/content/tani_corpus/tani_corpus.json'
    if os.path.exists(src) and not os.path.exists(dst):                # local copy: much faster to read
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
    if os.path.exists(dst):
        os.environ.setdefault('MTANI_CORPUS_PATH', dst)
    print('Colab: bundle', os.environ['MTANI_BUNDLE_DIR'], '| installed', need or 'nothing')
else:
    print('Not on Colab: using local paths and the current Python environment.')

## Library L0 · Imports, settings, hardware

Settings live in `Config`; any field can be overridden as `MTANI_<FIELD>`. `MTANI_STAGE` (`prepare`, `train`,
`evaluate`, default `all`) selects which task cells do work when the notebook runs as a batch job.

In [ ]:
import os, sys, re, json, math, time, random, hashlib, bisect, platform, subprocess, unicodedata, warnings, shutil
from dataclasses import dataclass, field, asdict, fields
from collections import Counter, defaultdict

import numpy as np
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
import torch.nn.functional as F
import transformers
import sentencepiece as spm
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF, TER
import pandas as pd
from torch.utils.data import IterableDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig

warnings.filterwarnings('ignore', category=UserWarning, module='sacrebleu')
import logging
logging.getLogger('sacrebleu').setLevel(logging.ERROR)
transformers.logging.set_verbosity_error()

STAGE = os.environ.get('MTANI_STAGE', 'all').strip().lower()
def RUN(stage):
    return STAGE == 'all' or STAGE == stage


def _bundle():
    """The m2m-tani-base-57M bundle (corpus, exact split, leak-free test set, base-model results)."""
    for p in (os.environ.get('MTANI_BUNDLE_DIR'), '../m2m-tani-base-57M', 'm2m-tani-base-57M', '../../m2m-tani-base-57M'):
        if p and os.path.isdir(p):
            return os.path.abspath(p)
    return os.path.abspath('../m2m-tani-base-57M')


B = _bundle()


@dataclass
class Config:
    # --- run ---
    model_name: str = 'nllb-tani-1.3B'
    plm: str = 'facebook/nllb-200-distilled-1.3B'
    run_mode: str = 'benchmark'          # must match the base split manifest
    seed: int = 13
    device: str = 'auto'
    pad_multiple: int = 8
    corpus_path: str = os.path.join(B, '02_data', 'tani_corpus.json')
    manifest_path: str = os.path.join(B, '03_splits', 'split_manifest.json')
    leakfree_path: str = os.path.join(B, '07_eval_test_data_leakfree', 'leakfree_set', 'leakfree_testset.json')
    base_metrics_path: str = os.path.join(B, '05_eval_training_run_test_split', 'metrics.json')
    base_leakfree_metrics_path: str = os.path.join(B, '07_eval_test_data_leakfree', 'results', 'pytorch', 'metrics.json')
    run_dir: str = 'runs'
    ckpt_dir: str = 'checkpoints'
    debug_books: str = ''                # smoke tests only, e.g. "JHN,PHM"
    debug_max_nonbible: int = 0
    # --- languages / tokenizer ---
    init_from_langs: str = 'lus_Latn,kac_Latn'   # NLLB codes whose embeddings initialise the 7 new codes
    sample_temperature: float = 5.0
    # --- data stream (identical to m2mTani-base) ---
    mix_tani_eng: float = 0.35
    mix_tani_hin: float = 0.35
    mix_tani_tani: float = 0.20
    mix_eng_hin: float = 0.10
    dae_prob: float = 0.10
    dae_noise: float = 0.15
    versions_per_epoch: int = 3
    modern_english_targets: bool = True
    dictionary_weight: float = 0.3
    spm_alpha: float = 0.1
    max_len: int = 256
    max_len_ratio: float = 3.0
    max_tokens_per_batch: int = 1400     # set from GPU memory below unless given
    tokens_per_update: int = 24000
    num_workers: int = 1
    # --- optimisation ---
    lr: float = 1e-4                     # base: 5e-4 from scratch
    warmup_steps: int = 1000             # base: 4,000
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1
    rdrop_alpha: float = 5.0
    clip_norm: float = 1.0
    max_updates: int = 200_000
    eval_every: int = 2000
    patience: int = 5
    min_delta: float = 0.1
    keep_top_k: int = 5
    dev_per_direction: int = 40
    gradient_checkpointing: bool = True
    ckpt_dtype: str = 'bfloat16'
    save_every_minutes: float = 60.0
    job_hours: float = 3.7
    safety_minutes: float = 20.0
    # --- evaluation ---
    beam: int = 5
    eval_batch_size: int = 32
    test_per_direction: int = 500
    examples_to_show: int = 10

    def __post_init__(self):
        for f in fields(self):
            env = os.environ.get('MTANI_' + f.name.upper())
            if env is not None:
                cur = getattr(self, f.name)
                setattr(self, f.name, env.lower() in ('1', 'true', 'yes') if isinstance(cur, bool) else type(cur)(env))
        assert abs(self.mix_tani_eng + self.mix_tani_hin + self.mix_tani_tani + self.mix_eng_hin - 1) < 1e-6


cfg = Config()
RUN_DIR, CKPT_DIR = os.path.abspath(cfg.run_dir), os.path.abspath(cfg.ckpt_dir)
os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)

if cfg.device == 'cpu':
    DEVICE, AMP_DTYPE = torch.device('cpu'), None
elif torch.cuda.is_available() and cfg.device in ('auto', 'cuda'):
    DEVICE = torch.device('cuda', 0)
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available() and cfg.device in ('auto', 'mps'):
    DEVICE, AMP_DTYPE = torch.device('mps'), None
else:
    DEVICE, AMP_DTYPE = torch.device('cpu'), None

# micro-batch from GPU memory: the 256k-way output layer (twice, with R-Drop) dominates activation memory
if DEVICE.type == 'cuda':
    gpu_gib = torch.cuda.get_device_properties(DEVICE).total_memory / 2**30
    if 'MTANI_MAX_TOKENS_PER_BATCH' not in os.environ:
        cfg.max_tokens_per_batch = 2400 if gpu_gib >= 70 else 1400 if gpu_gib >= 38 else 800
    if 'MTANI_EVAL_BATCH_SIZE' not in os.environ:
        cfg.eval_batch_size = 48 if gpu_gib >= 70 else 32 if gpu_gib >= 38 else 16
try:
    ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
except (ValueError, OSError, AttributeError):
    ram_gib = 0
if 'MTANI_NUM_WORKERS' not in os.environ:
    cfg.num_workers = 2 if ram_gib >= 48 else 1

print(f'Stage: {STAGE} | PLM: {cfg.plm}')
print(f'Python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__} | '
      f'sentencepiece {spm.__version__} | sacrebleu {sacrebleu.__version__}')
print(f'Device: {DEVICE} | mixed precision: {AMP_DTYPE} | RAM {ram_gib:.0f} GiB')
print(f'Bundle: {B}')
for k in ('corpus_path', 'manifest_path', 'leakfree_path', 'base_metrics_path'):
    print(f'  {k:18} {"ok " if os.path.exists(getattr(cfg, k)) else "MISSING"} {getattr(cfg, k)}')
print(f'Batching: {cfg.max_tokens_per_batch:,} tokens per micro-batch, {cfg.tokens_per_update:,} target tokens per update, '
      f'eval batch {cfg.eval_batch_size}, {cfg.num_workers} data worker(s)')
json.dump(asdict(cfg), open(os.path.join(RUN_DIR, 'config.json'), 'w'), indent=2)

DEADLINE = time.time() + cfg.job_hours * 3600 - cfg.safety_minutes * 60
print(f'This job stops training at {time.strftime("%H:%M:%S", time.localtime(DEADLINE))} '
      f'({(DEADLINE - time.time()) / 3600:.2f} h from now)')

## Library L1 · Corpus (identical to m2mTani-base)

`get_corpus()` reads the corpus line by line into compact records and keeps it in memory.

In [ ]:
DIALECTS = ['adi', 'apt', 'adl', 'mrg', 'njz', 'tgj', 'hru']
HUBS = ['eng', 'hin']
LANGS = DIALECTS + HUBS
LANG_NAMES = {'adi': 'Adi', 'apt': 'Apatani', 'adl': 'Galo', 'mrg': 'Mising', 'njz': 'Nyishi', 'tgj': 'Tagin',
              'hru': 'Hruso', 'eng': 'English', 'hin': 'Hindi'}
NLLB_CODE = {'adi': 'adi_Latn', 'apt': 'apt_Latn', 'adl': 'adl_Latn', 'mrg': 'mrg_Latn', 'njz': 'njz_Latn',
             'tgj': 'tgj_Latn', 'hru': 'hru_Latn', 'eng': 'eng_Latn', 'hin': 'hin_Deva'}
FIXED_ENGLISH = ['engbsb', 'eng-web', 'engwebp', 'engoebcw', 'eng-asv', 'eng-kjv', 'original']   # eng->X input
ARCHAIC_ENGLISH = {'engWycliffe', 'engwyc2017', 'engwyc2018', 'engtnt', 'enggnv', 'eng-kjv', 'engkjvcpb', 'engwebster',
                   'engDRA', 'eng-asv', 'engasvbt', 'eng-rv', 'engDBY', 'engylt', 'engnoy'}


def iter_entry_lines(path):
    head, in_entries = [], False
    with open(path, encoding='utf-8') as f:
        for line in f:
            if not in_entries:
                if line.startswith('  "entries": ['):
                    in_entries = True
                    yield json.loads(''.join(head).rstrip().rstrip(',') + '}'), None
                elif not line.startswith('  "excluded"'):
                    head.append(line)
                continue
            s = line.strip()
            if s.startswith('{'):
                yield None, json.loads(s.rstrip(','))


ENT = META = GROUP = by_lang = None


def load_corpus():
    global ENT, META, GROUP, by_lang
    t0 = time.time()
    ENT, META = [], {}
    debug_books = set(filter(None, cfg.debug_books.split(',')))
    nonbible_kept = Counter()
    for meta, e in iter_entry_lines(cfg.corpus_path):
        if meta is not None:
            META = meta
            continue
        lang = e['lang'].split('_')[0]
        if lang not in LANGS:
            continue
        if debug_books:
            if 'book' in e:
                if e['book'] not in debug_books:
                    continue
            else:
                nonbible_kept[e['dataset']] += 1
                if nonbible_kept[e['dataset']] > cfg.debug_max_nonbible:
                    continue
        ENT.append({'id': e['id'], 'group': e['group'], 'lang': lang, 'text': e['text'], 'domain': e['domain'],
                    'kind': e.get('kind'), 'part': e.get('part'), 'book': e.get('book'), 'chapter': e.get('chapter'),
                    'flags': e.get('flags', []), 'dataset': e.get('dataset', ''), 'version': e.get('text_version'),
                    'tr': [(t['version'], t['text']) for t in e['translations']]})
    print(f'Loaded {len(ENT):,} entries in {time.time() - t0:.0f}s')
    assert len({e['id'] for e in ENT}) == len(ENT), 'duplicate entry ids'
    assert all(e['text'] and e['tr'] for e in ENT), 'entry without text or translations'
    by_lang = defaultdict(list)
    for i, e in enumerate(ENT):
        by_lang[e['lang']].append(i)
    GROUP = defaultdict(lambda: defaultdict(list))
    for i, e in enumerate(ENT):
        GROUP[e['group']][e['lang']].append(i)
    print(f'{"language":10} {"code":9} {"entries":>8} {"English tr.":>12}')
    for l in LANGS:
        if l != 'eng':
            print(f'{LANG_NAMES[l]:10} {NLLB_CODE[l]:9} {len(by_lang[l]):>8,} {sum(len(ENT[i]["tr"]) for i in by_lang[l]):>12,}')
    print(f'{len(GROUP):,} groups')


def get_corpus():
    if ENT is None:
        load_corpus()


def fixed_english(i):
    tr = dict(ENT[i]['tr'])
    return next((tr[v] for v in FIXED_ENGLISH if v in tr), ENT[i]['tr'][0][1])

## Library L2 · The exact split of m2mTani-base

The manifest is **loaded, never rebuilt**: its corpus SHA-256, run mode and seed must match, otherwise the notebook
stops (a new split would put the base model's test verses into this model's training data and make the comparison
meaningless). Rule: a group is dev/test if listed in `heldout_groups`, otherwise train.

In [ ]:
manifest = SPLIT = None


def corpus_sha():
    h = hashlib.sha256()
    with open(cfg.corpus_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()


def get_split():
    global manifest, SPLIT
    if SPLIT is not None:
        return
    get_corpus()
    manifest = json.load(open(cfg.manifest_path))
    if not cfg.debug_books:
        sha = corpus_sha()
        assert manifest['corpus_sha256'] == sha, f'corpus differs from the one the split was made for ({sha[:12]}…)'
    assert manifest['run_mode'] == cfg.run_mode and manifest['seed'] == cfg.seed, 'manifest made for another mode/seed'
    SPLIT = defaultdict(lambda: 'train')
    for s in ('dev', 'test'):
        for g in manifest['heldout_groups'][s]:
            SPLIT[g] = s
    shutil.copy(cfg.manifest_path, os.path.join(RUN_DIR, 'split_manifest.json'))   # provenance of this run


def split_of(i):
    return SPLIT[ENT[i]['group']]


def split_report():
    get_corpus(); get_split()
    assert not set(manifest['heldout_groups']['dev']) & set(manifest['heldout_groups']['test'])
    for l in LANGS:
        if l == 'eng':
            continue
        train_texts = {ENT[i]['text'] for i in by_lang[l] if split_of(i) == 'train'}
        leak = [i for i in by_lang[l] if split_of(i) != 'train' and ENT[i]['text'] in train_texts]
        assert not leak, f'{l}: {len(leak)} held-out texts also in train'
    groups = Counter(SPLIT[g] for g in GROUP)
    print(f"Exact base split: {groups['train']:,} train / {groups['dev']:,} dev / {groups['test']:,} test groups; "
          f"held-out chapters dev {len(manifest['heldout_chapters']['dev'])}, test {len(manifest['heldout_chapters']['test'])}")
    print(f'{"language":10} {"train":>8} {"dev":>7} {"test":>7}')
    for l in LANGS:
        if l != 'eng':
            c = Counter(split_of(i) for i in by_lang[l])
            print(f'{LANG_NAMES[l]:10} {c["train"]:>8,} {c["dev"]:>7,} {c["test"]:>7,}')

## Library L3 · NLLB tokenizer and the 7 new language codes

NLLB's SentencePiece model (`sentencepiece.bpe.model`) is used directly so that source-side **subword sampling
(α = 0.1)** works exactly as in the base recipe; pieces are mapped to NLLB ids through the Hugging Face tokenizer.
`eng_Latn` and `hin_Deva` exist; the seven Tani-region codes are added as special tokens. `<mask>` (denoising) is NLLB's
own mask token.

**Punctuation:** NLLB's vocabulary has no curly quotes or long dashes, so they are mapped to ASCII before encoding (as in
NLLB's own preprocessing); otherwise they become `<unk>`. References in the evaluation are left untouched (identical to
the base evaluation), so outputs with straight quotes lose a little chrF++ where a reference uses curly ones.

In [ ]:
TOK = SPM = LANG_ID = MASK_ID = None
PAD = EOS = BOS = UNK = None
NEW_CODES = []


def get_tokenizer():
    global TOK, SPM, LANG_ID, MASK_ID, PAD, EOS, BOS, UNK, NEW_CODES
    if TOK is not None:
        return
    from huggingface_hub import hf_hub_download
    TOK = AutoTokenizer.from_pretrained(cfg.plm)
    vocab = TOK.get_vocab()
    NEW_CODES = [c for c in NLLB_CODE.values() if c not in vocab]
    if NEW_CODES:
        TOK.add_tokens(NEW_CODES, special_tokens=True)
    SPM = spm.SentencePieceProcessor(model_file=hf_hub_download(cfg.plm, 'sentencepiece.bpe.model'))
    LANG_ID = {l: TOK.convert_tokens_to_ids(c) for l, c in NLLB_CODE.items()}
    PAD, EOS, BOS, UNK = TOK.pad_token_id, TOK.eos_token_id, TOK.bos_token_id, TOK.unk_token_id
    MASK_ID = TOK.mask_token_id
    assert all(v is not None and v != UNK for v in LANG_ID.values()), LANG_ID
    assert MASK_ID is not None and MASK_ID != UNK
    print(f'Tokenizer: {len(TOK):,} tokens; new codes {NEW_CODES}; language ids {LANG_ID}; pad {PAD} eos {EOS} mask {MASK_ID}')


# NLLB's vocabulary has no curly quotes or long dashes (its training data was punctuation-normalised): map them to
# ASCII first, as NLLB's own preprocessing does, or every one becomes <unk> (1.5 % of Mising pieces, 4 % of English).
PUNCT = str.maketrans({'\u201c': '"', '\u201d': '"', '\u201e': '"', '\u00ab': '"', '\u00bb': '"', '\u2018': "'",
                       '\u2019': "'", '\u201a': "'", '\u2013': '-', '\u2014': ' - ', '\u2026': '...'})


def encode_text(text, sample=False):
    """NLLB ids for `text` (no language code, no </s>); '<mask>' becomes the mask token; optional subword sampling."""
    ids = []
    text = text.translate(PUNCT)
    for k, part in enumerate(text.split('<mask>')):
        if k:
            ids.append(MASK_ID)
        part = part.strip()
        if part:
            pieces = SPM.encode(part, out_type=str, enable_sampling=sample, alpha=cfg.spm_alpha, nbest_size=-1)
            ids += TOK.convert_tokens_to_ids(pieces)
    return ids


def decode_ids(ids):
    return TOK.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()


def tokenizer_report():
    get_corpus(); get_split(); get_tokenizer()
    sample = [ENT[by_lang[l][k]]['text'] for l in LANGS if l != 'eng' for k in range(0, min(200, len(by_lang[l])), 20)]
    agree = sum(encode_text(s) == TOK(s.translate(PUNCT), add_special_tokens=False)['input_ids'] for s in sample)
    print(f'SentencePiece path agrees with the Hugging Face tokenizer on {agree}/{len(sample)} sample sentences '
          f'(both after punctuation normalisation)')
    print(f'\n{"language":10} {"pieces/word":>12} {"unk %":>7}   (base m2mTani tokenizer, for reference: '
          f'tgj 1.19 · adl 1.29 · njz 1.35 · hin 1.39 · adi 1.65 · hru 2.08 · apt 2.26 · mrg 2.52)')
    for l in LANGS:
        texts = ([fixed_english(i) for i in random.Random(1).sample(range(len(ENT)), min(2000, len(ENT)))] if l == 'eng'
                 else [ENT[i]['text'] for i in by_lang[l][:2000] if split_of(i) == 'train'])
        if texts:
            ids = [encode_text(t) for t in texts]
            n = sum(map(len, ids))
            print(f'{LANG_NAMES[l]:10} {n / max(1, sum(len(t.split()) for t in texts)):>12.2f} '
                  f'{100 * sum(x == UNK for s in ids for x in s) / max(1, n):>6.2f}%')

## Library L4 · Streaming balanced dataset and denoising (identical recipe)

Same mixture, temperature sampling, English re-sampling, modern-English targets, dictionary weighting, denoising and
length-bucketed batches as m2mTani-base. Only the encoding differs: **NLLB convention** — source = `[src code] text </s>`,
target = `[tgt code] text </s>`; denoising examples use the same code on both sides.

In [ ]:
TRAIN_POOLS = None


def temperature_weights(sizes, T):
    w = {k: (n ** (1.0 / T)) for k, n in sizes.items() if n > 0}
    z = sum(w.values())
    return {k: v / z for k, v in w.items()}


def build_pools(which):
    idx = [i for i in range(len(ENT)) if split_of(i) == which]
    t2e = {l: [i for i in idx if ENT[i]['lang'] == l] for l in DIALECTS}
    t2h, t2t = defaultdict(list), defaultdict(list)
    eh = [i for i in idx if ENT[i]['lang'] == 'hin']
    for g, d in GROUP.items():
        if SPLIT[g] != which:
            continue
        tani = [l for l in d if l in DIALECTS]
        if 'hin' in d:
            for l in tani:
                t2h[l].append(g)
        for a in tani:
            for b in tani:
                if a < b:
                    t2t[(a, b)].append(g)
    return {'t2e': {k: v for k, v in t2e.items() if v}, 't2h': dict(t2h), 't2t': dict(t2t), 'eh': eh}


def weight_of(i):
    return cfg.dictionary_weight if ENT[i]['kind'] == 'word' else 1.0


class PairStream(IterableDataset):
    def __init__(self, pools, seed, train=True):
        self.pools, self.seed, self.train = pools, seed, train
        self.cum = {('t2e', l): list(np.cumsum([weight_of(i) for i in idx])) for l, idx in pools['t2e'].items()}
        self.w_t2e = temperature_weights({l: len(v) for l, v in pools['t2e'].items()}, cfg.sample_temperature)
        self.w_t2h = temperature_weights({l: len(v) for l, v in pools['t2h'].items()}, cfg.sample_temperature)
        self.w_t2t = temperature_weights({k: len(v) for k, v in pools['t2t'].items()}, cfg.sample_temperature)
        self.w_dae = self.w_t2e
        mix = {'t2e': cfg.mix_tani_eng if self.w_t2e else 0, 't2h': cfg.mix_tani_hin if self.w_t2h else 0,
               't2t': cfg.mix_tani_tani if self.w_t2t else 0, 'eh': cfg.mix_eng_hin if pools['eh'] else 0}
        z = sum(mix.values())
        self.mix = {k: v / z for k, v in mix.items() if v}
        self.epoch_len = max(1, 2 * sum(len(v) for v in pools['t2e'].values()))
        self.stats = Counter()

    def pick(self, rng, weights):
        keys = list(weights)
        return keys[bisect.bisect(list(np.cumsum([weights[k] for k in keys])), rng.random() * 0.999999)]

    def entry(self, rng, lang):
        idx, cum = self.pools['t2e'][lang], self.cum[('t2e', lang)]
        return idx[min(len(idx) - 1, bisect.bisect(cum, rng.random() * cum[-1]))]

    def english(self, rng, i, epoch, as_target=False):
        tr = ENT[i]['tr']
        if as_target and cfg.modern_english_targets:
            modern = [x for x in tr if x[0] not in ARCHAIC_ENGLISH
                      and not (x[0] == 'original' and ENT[i]['dataset'] == 'adi-english-bible')]   # Adi's is the ASV
            tr = modern or tr
        if len(tr) <= cfg.versions_per_epoch:
            return rng.choice(tr)[1]
        er = random.Random(hash((self.seed, i, epoch)) & 0xffffffff)
        return tr[er.choice(er.sample(range(len(tr)), cfg.versions_per_epoch))][1]

    def hindi(self, rng, g):
        return ENT[rng.choice(GROUP[g]['hin'])]['text']

    def noise(self, rng, text):
        out = []
        for w in text.split():
            r = rng.random()
            if r < cfg.dae_noise / 2:
                continue
            out.append('<mask>' if r < cfg.dae_noise else w)
        return ' '.join(out) or text

    def example(self, rng, n):
        epoch = n // self.epoch_len
        if self.train and rng.random() < cfg.dae_prob:
            l = self.pick(rng, self.w_dae)
            t = ENT[self.entry(rng, l)]['text']
            return 'dae', l, l, self.noise(rng, t), t
        pool = self.pick(rng, self.mix)
        to_eng = rng.random() < 0.5
        if pool == 't2e':
            l = self.pick(rng, self.w_t2e)
            i = self.entry(rng, l)
            pair = (l, ENT[i]['text'], 'eng', self.english(rng, i, epoch, as_target=to_eng))
        elif pool == 't2h':
            l = self.pick(rng, self.w_t2h)
            g = rng.choice(self.pools['t2h'][l])
            pair = (l, ENT[rng.choice(GROUP[g][l])]['text'], 'hin', self.hindi(rng, g))
        elif pool == 't2t':
            a, b = self.pick(rng, self.w_t2t)
            g = rng.choice(self.pools['t2t'][(a, b)])
            pair = (a, ENT[rng.choice(GROUP[g][a])]['text'], b, ENT[rng.choice(GROUP[g][b])]['text'])
        else:
            i = rng.choice(self.pools['eh'])
            pair = ('hin', ENT[i]['text'], 'eng', self.english(rng, i, epoch, as_target=to_eng))
        if not to_eng:
            pair = (pair[2], pair[3], pair[0], pair[1])
        return pool, pair[0], pair[2], pair[1], pair[3]

    def encode(self, rng, kind, sl, tl, src, tgt):
        s = encode_text(src, sample=self.train and cfg.spm_alpha > 0)
        t = encode_text(tgt)
        if len(s) + 2 > cfg.max_len or len(t) + 2 > cfg.max_len:
            self.stats['dropped_too_long'] += 1
            return None
        if max(len(s), len(t)) > 10 and max(len(s), len(t)) / max(1, min(len(s), len(t))) > cfg.max_len_ratio:
            self.stats['dropped_ratio'] += 1
            return None
        return [LANG_ID[sl]] + s + [EOS], [LANG_ID[tl]] + t + [EOS], f'{sl}>{tl}'      # NLLB: source code, target code

    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        rng = random.Random(self.seed * 1000 + (info.id if info else 0))
        n, buf = 0, []
        while True:
            while len(buf) < 4096:
                ex = self.encode(rng, *self.example(rng, n))
                n += 1
                if ex:
                    buf.append(ex)
            buf.sort(key=lambda x: (len(x[0]), len(x[1])))
            batches, cur, width = [], [], 0
            for ex in buf:
                w = max(width, len(ex[0]) + len(ex[1]))
                if cur and w * (len(cur) + 1) > cfg.max_tokens_per_batch:
                    batches.append(cur); cur, width = [], 0
                    w = len(ex[0]) + len(ex[1])
                cur.append(ex); width = w
            rng.shuffle(batches)
            buf = cur
            for b in batches:
                yield collate(b)


def round_up(n):
    return -(-n // cfg.pad_multiple) * cfg.pad_multiple


def collate(batch):
    ls, lt = round_up(max(len(b[0]) for b in batch)), round_up(max(len(b[1]) for b in batch))
    src = torch.full((len(batch), ls), PAD, dtype=torch.long)
    lab = torch.full((len(batch), lt), -100, dtype=torch.long)
    for k, (s, t, _) in enumerate(batch):
        src[k, :len(s)] = torch.tensor(s)
        lab[k, :len(t)] = torch.tensor(t)
    return {'input_ids': src, 'attention_mask': (src != PAD).long(), 'labels': lab, 'directions': [b[2] for b in batch]}


def get_pools():
    global TRAIN_POOLS
    if TRAIN_POOLS is None:
        get_corpus(); get_split(); get_tokenizer()
        TRAIN_POOLS = build_pools('train')


def stream_preview():
    get_pools()
    print('Training pools:')
    print('  dialect<->English entries:', {l: len(v) for l, v in TRAIN_POOLS['t2e'].items()})
    print('  dialect<->Hindi groups:  ', {l: len(v) for l, v in TRAIN_POOLS['t2h'].items()})
    print('  dialect<->dialect groups:', {f'{a}-{b}': len(v) for (a, b), v in sorted(TRAIN_POOLS['t2t'].items())})
    print('  English<->Hindi entries: ', len(TRAIN_POOLS['eh']))
    peek = PairStream(TRAIN_POOLS, cfg.seed)
    rng0 = random.Random(0)
    for _ in range(6):
        k, sl, tl, s, t = peek.example(rng0, 0)
        print(f'  [{k:3}] {NLLB_CODE[sl]} {s[:55]!r} -> {NLLB_CODE[tl]} {t[:55]!r}')
    cnt = Counter(peek.example(rng0, 0)[0] for _ in range(20000))
    print('  sampled mixture:', {k: round(v / 20000, 3) for k, v in cnt.items()})
    b = next(iter(peek))
    print(f"  first batch: {b['input_ids'].shape[0]} examples, source width {b['input_ids'].shape[1]}, "
          f"target width {b['labels'].shape[1]}; e.g. {decode_ids(b['input_ids'][0].tolist())[:70]!r}")

## Library L5 · NLLB-200 distilled 1.3B with the new language codes

The embedding matrix (tied encoder/decoder/output) is resized for the 7 new codes; each new row starts from the mean of
the `init_from_langs` codes (Mizo, Jingpho) plus noise at 1 % of its scale, so the codes start close to related
Tibeto-Burman Latin-script languages but are distinguishable.

In [ ]:
def load_plm():
    get_tokenizer()
    mdl = AutoModelForSeq2SeqLM.from_pretrained(cfg.plm).float()
    if len(TOK) > mdl.get_input_embeddings().weight.shape[0]:
        mdl.resize_token_embeddings(len(TOK), pad_to_multiple_of=8)
    emb = mdl.get_input_embeddings().weight
    with torch.no_grad():
        src_ids = [TOK.convert_tokens_to_ids(c) for c in cfg.init_from_langs.split(',')]
        assert all(i != UNK for i in src_ids), cfg.init_from_langs
        base = emb[src_ids].mean(0)
        g = torch.Generator().manual_seed(cfg.seed)
        for c in NEW_CODES:
            emb[TOK.convert_tokens_to_ids(c)] = base + 0.01 * base.std() * torch.randn(base.shape, generator=g)
    mdl.config.name_or_path = cfg.model_name
    return mdl


def model_summary():
    mdl = load_plm()
    n = sum(p.numel() for p in mdl.parameters())
    c = mdl.config
    print(f'{cfg.plm}: {n / 1e6:,.0f}M parameters | vocab {c.vocab_size:,} (+{len(NEW_CODES)} codes) | d_model {c.d_model} | '
          f'{c.encoder_layers}+{c.decoder_layers} layers | dropout {c.dropout}')
    tied = mdl.get_output_embeddings().weight.data_ptr() == mdl.get_input_embeddings().weight.data_ptr()
    print('output layer tied to embeddings:', tied)
    del mdl

## Library L6 · Fine-tuning loop (same recipe as m2mTani-base)

AdamW (β 0.9/0.98, ε 1e-8, weight decay 1e-4 except norms and biases), linear warm-up then inverse-square-root decay,
gradient clipping 1.0, label smoothing 0.1, **R-Drop** (batch stacked twice, symmetric KL, α 5), bf16 autocast,
accumulation to ~24k target tokens per update. Every `eval_every` updates: greedy dev chrF++ in every direction, top-k
checkpoints kept, early stopping. The full state is saved regularly and the next run resumes. At the end the top-k
checkpoints are averaged and compared with the best single one on beam-5 dev chrF++; the better goes to
`checkpoints/nllb-tani-1.3B-final/`. Single GPU (the base notebook's multi-GPU launcher is not used here).

In [ ]:
FINAL_DIR = os.path.join(CKPT_DIR, 'nllb-tani-1.3B-final')
STATE_PT = os.path.join(CKPT_DIR, 'last', 'train_state.pt')
TRAIN_JSON = os.path.join(RUN_DIR, 'train_state.json')
chrf_metric = CHRF(word_order=2)


def eval_items(which, per_direction, seed):
    """{(src,tgt): [(src_text, [refs], meta)]} for every direction with data in split `which` (as in the base)."""
    rng = random.Random(seed)
    items = defaultdict(list)
    for g in sorted(g for g in GROUP if SPLIT[g] == which):
        d = GROUP[g]
        tani = [l for l in d if l in DIALECTS]
        hin = [ENT[i]['text'] for i in d.get('hin', [])]
        hin_irv = next((ENT[i]['text'] for i in d.get('hin', []) if ENT[i]['version'] == 'hin2017'), hin[0] if hin else None)
        rep = next(i for idx in d.values() for i in idx)
        eng_refs = list(dict.fromkeys(t for idx in d.values() for i in idx for _, t in ENT[i]['tr']))
        meta = {'group': g, 'part': ENT[rep]['part'] or f"non-Bible ({ENT[rep]['domain']})"}
        eng_src = fixed_english(rep)
        for l in tani:
            texts = [ENT[i]['text'] for i in d[l]]
            refs_eng = list(dict.fromkeys(t for i in d[l] for _, t in ENT[i]['tr']))
            items[(l, 'eng')].append((texts[0], refs_eng, meta))
            items[('eng', l)].append((fixed_english(d[l][0]), texts, meta))
            if hin:
                items[(l, 'hin')].append((texts[0], [hin_irv], meta))
                items[('hin', l)].append((hin_irv, texts, meta))
            for m in tani:
                if m != l:
                    items[(l, m)].append((texts[0], [ENT[i]['text'] for i in d[m]], meta))
        if hin and g.startswith('bible:'):
            items[('eng', 'hin')].append((eng_src, [hin_irv], meta))
            items[('hin', 'eng')].append((hin_irv, eng_refs, meta))
    out = {}
    for k, v in items.items():
        rng.shuffle(v)
        out[k] = v[:per_direction] if per_direction else v
    return out


@torch.no_grad()
def translate_batch(mdl, texts, src, tgt, beam=1, batch_size=None):
    """NLLB convention: [src code] text </s> in, target code forced as the first decoder token."""
    mdl.eval()
    batch_size = batch_size or cfg.eval_batch_size
    order = sorted(range(len(texts)), key=lambda k: len(texts[k]))
    out = [None] * len(texts)
    for s in range(0, len(order), batch_size):
        chunk = order[s:s + batch_size]
        ids = [[LANG_ID[src]] + encode_text(texts[k])[:cfg.max_len - 2] + [EOS] for k in chunk]
        L = round_up(max(map(len, ids)))
        x = torch.full((len(ids), L), PAD, dtype=torch.long)
        for r, row in enumerate(ids):
            x[r, :len(row)] = torch.tensor(row)
        x = x.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            gen = mdl.generate(input_ids=x, attention_mask=(x != PAD).long(), num_beams=beam, use_cache=True,
                               max_new_tokens=int(1.5 * L + 10), forced_bos_token_id=LANG_ID[tgt], length_penalty=1.0,
                               early_stopping=beam > 1, do_sample=False)
        for r, k in enumerate(chunk):
            out[k] = decode_ids(gen[r].tolist())
    return out


def corpus_chrf(hyps, refs):
    n = max(map(len, refs))
    return chrf_metric.corpus_score(hyps, [[r[k] if k < len(r) else None for r in refs] for k in range(n)]).score


def dev_score(mdl, dev, beam=1):
    scores = {}
    for s, t in sorted(dev):
        items = dev[(s, t)]
        hyps = translate_batch(mdl, [x[0] for x in items], s, t, beam=beam)
        scores[f'{s}>{t}'] = corpus_chrf(hyps, [x[1] for x in items])
    return float(np.mean(list(scores.values()))) if scores else 0.0, scores


def free_gpu_cache():
    import gc
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    elif DEVICE.type == 'mps':
        torch.mps.empty_cache()


def lr_at(step):
    step = max(step, 1)
    return cfg.lr * min(step / cfg.warmup_steps, math.sqrt(cfg.warmup_steps / step))


def make_optimizer(mdl):
    decay, no_decay = [], []
    for n, p in mdl.named_parameters():
        (no_decay if p.ndim < 2 or 'norm' in n.lower() or n.endswith('bias') else decay).append(p)
    return torch.optim.AdamW([{'params': decay, 'weight_decay': cfg.weight_decay},
                              {'params': no_decay, 'weight_decay': 0.0}], lr=cfg.lr, betas=(0.9, 0.98), eps=1e-8,
                             fused=DEVICE.type == 'cuda')


def shift_right(labels):
    dec = labels.new_full(labels.shape, PAD)
    dec[:, 1:] = labels[:, :-1].clone()
    dec[:, 0] = EOS                                            # NLLB's decoder starts from </s>
    return dec.masked_fill(dec == -100, PAD)


def forward_loss(mdl, batch):
    x, m, y = batch['input_ids'], batch['attention_mask'], batch['labels']
    dec = shift_right(y)
    mask = y != -100
    ntok = mask.sum()
    rdrop = cfg.rdrop_alpha > 0
    if rdrop:
        x, m, dec = torch.cat([x, x]), torch.cat([m, m]), torch.cat([dec, dec])
    with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
        logits = mdl(input_ids=x, attention_mask=m, decoder_input_ids=dec, use_cache=False).logits
    lp_all = F.log_softmax(logits.float(), -1)
    del logits
    lps = lp_all.chunk(2) if rdrop else (lp_all,)
    ces = [F.cross_entropy(lp.reshape(-1, lp.size(-1)), y.reshape(-1), ignore_index=-100,
                           label_smoothing=cfg.label_smoothing, reduction='sum') for lp in lps]
    if not rdrop:
        return ces[0], ntok
    lp, lp2 = lps
    kl = (F.kl_div(lp, lp2, log_target=True, reduction='none').sum(-1) +
          F.kl_div(lp2, lp, log_target=True, reduction='none').sum(-1))
    kl = (kl * mask).sum() / 2
    return (ces[0] + ces[1]) / 2 + cfg.rdrop_alpha * kl, ntok


def save_state(mdl, opt, scaler, st):
    os.makedirs(os.path.dirname(STATE_PT), exist_ok=True)
    tmp = STATE_PT + '.tmp'
    torch.save({'model': mdl.state_dict(), 'optimizer': opt.state_dict(), 'scaler': scaler.state_dict() if scaler else None,
                'state': st, 'rng': {'python': random.getstate(), 'numpy': np.random.get_state(), 'torch': torch.get_rng_state()}}, tmp)
    os.replace(tmp, STATE_PT)
    json.dump({k: v for k, v in st.items() if k != 'history'}, open(TRAIN_JSON, 'w'), indent=1)


def save_checkpoint(mdl, path):
    dt = getattr(torch, cfg.ckpt_dtype)
    torch.save({k: v.detach().to('cpu', dt) for k, v in mdl.state_dict().items()}, path)


def average_checkpoints(paths):
    avg = None
    for p in paths:
        sd = torch.load(p, map_location='cpu', weights_only=True)
        if avg is None:
            avg = {k: v.float().clone() for k, v in sd.items()}
        else:
            for k in avg:
                avg[k] += sd[k].float()
    return {k: v / len(paths) for k, v in avg.items()}


def save_final(sd, dev_chrf):
    mdl = load_plm()
    mdl.load_state_dict(sd)
    os.makedirs(FINAL_DIR, exist_ok=True)
    mdl.save_pretrained(FINAL_DIR)
    GenerationConfig(decoder_start_token_id=EOS, eos_token_id=EOS, pad_token_id=PAD, bos_token_id=BOS,
                     num_beams=cfg.beam, max_length=cfg.max_len).save_pretrained(FINAL_DIR)
    TOK.save_pretrained(FINAL_DIR)
    json.dump({'model': cfg.model_name, 'base': cfg.plm,
               'languages': {l: {'name': LANG_NAMES[l], 'code': NLLB_CODE[l], 'id': LANG_ID[l], 'new': NLLB_CODE[l] in NEW_CODES}
                             for l in LANGS},
               'usage': 'source: [src code] text </s>; generate with forced_bos_token_id = target code',
               'dev_chrf++': dev_chrf, 'licence': 'CC BY-NC 4.0 (inherited from NLLB-200)'},
              open(os.path.join(FINAL_DIR, 'languages.json'), 'w'), indent=1)
    print(f'Saved {cfg.model_name} to {FINAL_DIR}')


def accumulation_steps():
    it = iter(PairStream(TRAIN_POOLS, cfg.seed + 4242))
    avg = float(np.mean([int((next(it)['labels'] != -100).sum()) for _ in range(30)]))
    return max(1, round(cfg.tokens_per_update / avg))


def train():
    st0 = json.load(open(TRAIN_JSON)) if os.path.exists(TRAIN_JSON) else {}
    if st0.get('finished') and os.path.isdir(FINAL_DIR):
        print('Training already finished:', FINAL_DIR)
        return
    mdl = load_plm().to(DEVICE)
    if cfg.gradient_checkpointing:
        mdl.gradient_checkpointing_enable()
    mdl.config.use_cache = False
    opt = make_optimizer(mdl)
    scaler = torch.amp.GradScaler('cuda') if AMP_DTYPE == torch.float16 else None
    st = {'step': 0, 'best': [], 'history': [], 'bad_evals': 0, 'best_score': -1.0, 'finished': False,
          'tokens': 0, 'train_seconds': 0.0}
    if os.path.exists(STATE_PT):
        ck = torch.load(STATE_PT, map_location='cpu', weights_only=False)
        mdl.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer'])
        if scaler and ck['scaler']:
            scaler.load_state_dict(ck['scaler'])
        st = ck['state']
        random.setstate(ck['rng']['python']); np.random.set_state(ck['rng']['numpy']); torch.set_rng_state(ck['rng']['torch'])
        del ck
        print(f"Resumed at update {st['step']:,} (best dev chrF++ {st['best_score']:.2f})")
    dev = eval_items('dev', cfg.dev_per_direction, cfg.seed + 7)
    print(f'Dev monitor: {len(dev)} directions, {sum(map(len, dev.values())):,} sentences')
    if st['step'] == 0:                                   # the starting point, for the record
        mean, _ = dev_score(mdl, dev); free_gpu_cache()
        print(f'  dev chrF++ before fine-tuning (greedy): {mean:.2f}')
        st['history'].append({'step': 0, 'dev_chrf++': mean})
    stream = PairStream(TRAIN_POOLS, cfg.seed + st['step'])
    workers = cfg.num_workers if platform.system() == 'Linux' else 0
    if workers:
        import gc
        gc.collect(); gc.freeze()
    loader = DataLoader(stream, batch_size=None, num_workers=workers, persistent_workers=workers > 0,
                        prefetch_factor=4 if workers else None, multiprocessing_context='fork' if workers else None)
    it = iter(loader)
    if st.get('accum') is None:
        st['accum'] = accumulation_steps()
    print(f"{st['accum']} micro-batches per update; R-Drop {'on (alpha ' + str(cfg.rdrop_alpha) + ')' if cfg.rdrop_alpha > 0 else 'off'}; "
          f"gradient checkpointing {cfg.gradient_checkpointing}; early stopping after {cfg.patience} evaluations without {cfg.min_delta}")
    last_save, t_start, tok_window = time.time(), time.time(), 0
    log = open(os.path.join(RUN_DIR, 'train_log.csv'), 'a')
    tb = None
    try:
        from torch.utils.tensorboard import SummaryWriter
        tb = SummaryWriter(os.path.join(RUN_DIR, 'tb'))
    except Exception:
        pass
    stop_reason = None
    mdl.train()
    while stop_reason is None:
        opt.zero_grad(set_to_none=True)
        toks, loss_sum = 0, 0.0
        for _ in range(st['accum']):
            b = next(it)
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in b.items() if k != 'directions'}
            loss, n = forward_loss(mdl, batch)
            (scaler.scale(loss) if scaler else loss).backward()
            toks += int(n); loss_sum += float(loss.detach())
        for g in opt.param_groups:
            g['lr'] = lr_at(st['step'] + 1)
        if scaler:
            scaler.unscale_(opt)
        for p in mdl.parameters():
            if p.grad is not None:
                p.grad.mul_(1.0 / max(1, toks))
        gn = torch.nn.utils.clip_grad_norm_(mdl.parameters(), cfg.clip_norm)
        if scaler:
            scaler.step(opt); scaler.update()
        else:
            opt.step()
        st['step'] += 1; st['tokens'] += toks; tok_window += toks
        if st['step'] % 10 == 0:
            el = time.time() - t_start
            print(f"step {st['step']:>6,} | loss/token {loss_sum / max(1, toks):.3f} | lr {lr_at(st['step']):.2e} | "
                  f"grad {float(gn):.2f} | {tok_window / max(1e-9, el):,.0f} tok/s", flush=True)
            log.write(f"{st['step']},{loss_sum / max(1, toks):.4f},{lr_at(st['step']):.3e},{float(gn):.3f}\n"); log.flush()
            if tb:
                tb.add_scalar('train/loss', loss_sum / max(1, toks), st['step']); tb.add_scalar('train/lr', lr_at(st['step']), st['step'])
            st['train_seconds'] += el; t_start, tok_window = time.time(), 0
        if st['step'] % cfg.eval_every == 0:
            mean, per = dev_score(mdl, dev)
            mdl.train(); free_gpu_cache()
            st['history'].append({'step': st['step'], 'dev_chrf++': mean})
            print(f"  dev chrF++ (greedy, {len(per)} directions): {mean:.2f}", flush=True)
            if tb:
                tb.add_scalar('dev/chrF++', mean, st['step'])
            path = os.path.join(CKPT_DIR, f"step_{st['step']:07d}.pt")
            save_checkpoint(mdl, path)
            st['best'] = sorted(st['best'] + [(mean, path)], reverse=True)
            for _, p in st['best'][cfg.keep_top_k:]:
                if os.path.exists(p):
                    os.remove(p)
            st['best'] = st['best'][:cfg.keep_top_k]
            if mean > st['best_score'] + cfg.min_delta:
                st['best_score'], st['bad_evals'] = mean, 0
            else:
                st['bad_evals'] += 1
                print(f"  no gain of {cfg.min_delta} for {st['bad_evals']}/{cfg.patience} evaluations")
            if st['bad_evals'] >= cfg.patience:
                stop_reason = 'early stopping'
        if stop_reason is None and st['step'] >= cfg.max_updates:
            stop_reason = 'max_updates reached'
        if stop_reason is None and time.time() > DEADLINE:
            stop_reason = 'job time'
        if stop_reason or time.time() - last_save > cfg.save_every_minutes * 60:
            save_state(mdl, opt, scaler, st)
            last_save = time.time()
    log.close()
    if tb:
        tb.close()
    if stop_reason == 'job time':
        print(f"Stopped for the time limit at update {st['step']:,}; the next run resumes from {STATE_PT}.")
        return
    print(f"Training finished at update {st['step']:,}: {stop_reason}.")
    if not st['best']:
        path = os.path.join(CKPT_DIR, f"step_{st['step']:07d}.pt"); save_checkpoint(mdl, path)
        st['best'] = [(dev_score(mdl, dev)[0], path)]
    del opt; free_gpu_cache()
    dev_beam = eval_items('dev', max(10, cfg.dev_per_direction // 2), cfg.seed + 11)
    best_single = torch.load(st['best'][0][1], map_location='cpu', weights_only=True)
    avg = average_checkpoints([p for _, p in st['best']])
    scores = {}
    for name, sd in (('best single', best_single), (f'average of {len(st["best"])}', avg)):
        mdl.load_state_dict(sd)
        scores[name] = dev_score(mdl, dev_beam, beam=cfg.beam)[0]
        free_gpu_cache()
        print(f'  {name}: dev chrF++ (beam {cfg.beam}) {scores[name]:.2f}')
    choice = max(scores, key=scores.get)
    del mdl; free_gpu_cache()
    save_final(avg if choice.startswith('average') else best_single, scores[choice])
    st['finished'], st['final'] = True, {'choice': choice, 'dev_chrf++_beam': scores[choice]}
    json.dump({k: v for k, v in st.items() if k != 'history'}, open(TRAIN_JSON, 'w'), indent=1)
    json.dump(st['history'], open(os.path.join(RUN_DIR, 'dev_history.json'), 'w'), indent=1)


def task_train():
    get_corpus(); get_split(); get_tokenizer(); get_pools()
    train()

## Library L7 · Evaluation on the held-out test chapters (same items and metrics as m2mTani-base)

Same test items (`test_per_direction` 500, seed + 3), beam 5, chrF++ (primary), multi-reference BLEU, TER, CER, length
ratio, copy-the-source baseline, and copy / off-target (character 3-gram language ID) / empty / hallucination rates.
Predictions are cached per direction, so an interrupted evaluation resumes. The last table compares with
m2mTani-base direction by direction.

In [ ]:
PRED_DIR = os.path.join(RUN_DIR, 'test_predictions')


def levenshtein(a, b):
    if not a:
        return len(b)
    if not b:
        return len(a)
    peq = defaultdict(int)
    for k, ch in enumerate(a):
        peq[ch] |= 1 << k
    m, pv, mv, score = len(a), (1 << len(a)) - 1, 0, len(a)
    high = 1 << (m - 1)
    for ch in b:
        eq = peq[ch]
        xv = eq | mv
        xh = (((eq & pv) + pv) ^ pv) | eq
        ph = mv | ~(xh | pv)
        mh = pv & xh
        if ph & high:
            score += 1
        elif mh & high:
            score -= 1
        ph = (ph << 1) | 1
        mh <<= 1
        pv = mh | ~(xv | ph)
        mv = ph & xv
        pv &= (1 << m) - 1
        mv &= (1 << m) - 1
    return score


class CharLID:
    """Naive-Bayes language identifier on character 3-grams, trained on training-split text."""

    def __init__(self, per_lang=20000, seed=5):
        rng = random.Random(seed)
        texts = defaultdict(list)
        for i, e in enumerate(ENT):
            if split_of(i) == 'train':
                texts[e['lang']].append(e['text'])
        texts['eng'] = [ENT[i]['tr'][0][1] for i in rng.sample(range(len(ENT)), min(len(ENT), per_lang * 2)) if split_of(i) == 'train']
        self.logp, self.default = {}, {}
        for l, ts in texts.items():
            ts = rng.sample(ts, min(per_lang, len(ts)))
            c = Counter(g for t in ts for g in self.grams(t))
            z = sum(c.values()) + len(c) + 1
            self.logp[l] = {g: math.log((n + 1) / z) for g, n in c.items()}
            self.default[l] = math.log(1 / z)

    @staticmethod
    def grams(t):
        t = f' {t.lower()} '
        return [t[k:k + 3] for k in range(len(t) - 2)]

    def __call__(self, t):
        gs = self.grams(t)
        return max(self.logp, key=lambda l: sum(self.logp[l].get(g, self.default[l]) for g in gs)) if gs else None


def md_table(header, rows):
    out = ['| ' + ' | '.join(map(str, header)) + ' |', '|' + '---|' * len(header)]
    return '\n'.join(out + ['| ' + ' | '.join(map(str, r)) + ' |' for r in rows])


def load_final():
    get_tokenizer()
    return AutoModelForSeq2SeqLM.from_pretrained(FINAL_DIR).to(DEVICE).eval()


def evaluate():
    if not os.path.isdir(FINAL_DIR):
        print(f'No fine-tuned model at {FINAL_DIR}: run T6 first.'); return None
    mdl = load_final()
    items = eval_items('test', cfg.test_per_direction, cfg.seed + 3)
    os.makedirs(PRED_DIR, exist_ok=True)
    lid = CharLID()
    bleu, chrf, ter = BLEU(), CHRF(word_order=2), TER()
    results, rows_csv = {}, []
    for (s, t), its in sorted(items.items()):
        key, cache = f'{s}>{t}', os.path.join(PRED_DIR, f'{s}-{t}.jsonl')
        if os.path.exists(cache):
            hyps = [json.loads(l)['hyp'] for l in open(cache, encoding='utf-8')]
        else:
            if time.time() > DEADLINE:
                print('Job deadline reached: run T7 again to finish the remaining directions.'); break
            hyps = translate_batch(mdl, [x[0] for x in its], s, t, beam=cfg.beam)
            free_gpu_cache()
            with open(cache + '.tmp', 'w', encoding='utf-8') as f:
                for h in hyps:
                    f.write(json.dumps({'hyp': h}, ensure_ascii=False) + '\n')
            os.replace(cache + '.tmp', cache)
        srcs, refs = [x[0] for x in its], [x[1] for x in its]
        n = max(map(len, refs))
        streams = [[r[k] if k < len(r) else None for r in refs] for k in range(n)]
        sb, sc, st_ = bleu.corpus_score(hyps, streams), chrf.corpus_score(hyps, streams), ter.corpus_score(hyps, streams)
        base = chrf.corpus_score(srcs, streams).score
        cer = sum(levenshtein(h, r[0]) for h, r in zip(hyps, refs)) / max(1, sum(len(r[0]) for r in refs))
        lr = sum(len(h.split()) for h in hyps) / max(1, sum(len(r[0].split()) for r in refs))
        sent = [chrf.sentence_score(h, r).score for h, r in zip(hyps, refs)]
        pred_lang = [lid(h) if h else None for h in hyps]
        diag = {'copy_rate': np.mean([h.strip() == x.strip() for h, x in zip(hyps, srcs)]),
                'off_target_rate': np.mean([p is not None and p != t for p in pred_lang]),
                'empty_rate': np.mean([not h.strip() for h in hyps]),
                'hallucinations': int(sum(len(h) > 2 * len(x) and c < 20 for h, x, c in zip(hyps, srcs, sent)))}
        results[key] = {'n': len(hyps), 'chrf++': sc.score, 'bleu': sb.score, 'ter': st_.score, 'cer': 100 * cer,
                        'length_ratio': lr, 'copy_baseline_chrf++': base, 'multi_reference': n > 1,
                        **{k: float(v) for k, v in diag.items()}}
        for x, h, c, p in zip(its, hyps, sent, pred_lang):
            rows_csv.append({'direction': key, 'group': x[2]['group'], 'part': x[2]['part'], 'source': x[0], 'hypothesis': h,
                             'reference': x[1][0], 'n_refs': len(x[1]), 'chrf++': round(c, 2), 'predicted_lang': p})
    signatures = {'bleu': str(bleu.get_signature()), 'chrf++': str(chrf.get_signature()), 'ter': str(ter.get_signature())}
    df = pd.DataFrame(rows_csv)
    df.to_csv(os.path.join(RUN_DIR, 'test_predictions.csv'), index=False)
    json.dump({'model': cfg.model_name, 'base': cfg.plm, 'split': 'test', 'signatures': signatures, 'directions': results,
               'mean_chrf++': float(np.mean([r['chrf++'] for r in results.values()])) if results else None},
              open(os.path.join(RUN_DIR, 'metrics.json'), 'w'), indent=1)
    return results, df


def kind_of(d):
    s, t = d.split('>')
    return ('dialect → English' if t == 'eng' and s in DIALECTS else 'English → dialect' if s == 'eng' and t in DIALECTS else
            'dialect → Hindi' if t == 'hin' and s in DIALECTS else 'Hindi → dialect' if s == 'hin' and t in DIALECTS else
            'dialect ↔ dialect' if s in DIALECTS and t in DIALECTS else 'English ↔ Hindi')


def show_results(results, df):
    rows = [[f'**{s}**'] + [('—' if s == t else f"{results[f'{s}>{t}']['chrf++']:.1f}" if f'{s}>{t}' in results else '·')
                            for t in LANGS] for s in LANGS]
    print('### chrF++ (rows = source, columns = target)\n'); print(md_table(['src \\ tgt'] + LANGS, rows))
    print(f"\nMean chrF++ over {len(results)} directions: {np.mean([r['chrf++'] for r in results.values()]):.2f}")
    if os.path.exists(cfg.base_metrics_path):                              # comparison with m2mTani-base
        base = json.load(open(cfg.base_metrics_path))['directions']
        common = sorted(set(base) & set(results))
        by = defaultdict(list)
        for d in common:
            by[kind_of(d)].append((results[d]['chrf++'], base[d]['chrf++'], results[d]['bleu'], base[d]['bleu']))
        out = [[k, len(v), f'{np.mean([x[1] for x in v]):.1f}', f'{np.mean([x[0] for x in v]):.1f}',
                f'{np.mean([x[0] - x[1] for x in v]):+.1f}', f'{np.mean([x[3] for x in v]):.1f}', f'{np.mean([x[2] for x in v]):.1f}']
               for k, v in sorted(by.items())]
        print('\n### NLLB fine-tuned vs m2mTani-base (same test items)\n')
        print(md_table(['direction type', 'directions', 'base chrF++', 'NLLB-ft chrF++', 'Δ', 'base BLEU', 'NLLB-ft BLEU'], out))
        wins = sum(results[d]['chrf++'] > base[d]['chrf++'] for d in common)
        print(f'\nNLLB-ft better in {wins}/{len(common)} directions; mean Δ chrF++ '
              f"{np.mean([results[d]['chrf++'] - base[d]['chrf++'] for d in common]):+.2f}")
    ex = []
    for d, sub in df.groupby('direction'):
        sub = sub.sort_values('chrf++')
        ex.append(pd.concat([sub.tail(cfg.examples_to_show).assign(rank='best'), sub.head(cfg.examples_to_show).assign(rank='worst')]))
    if ex:
        pd.concat(ex).to_csv(os.path.join(RUN_DIR, 'best_worst_examples.csv'), index=False)


def task_evaluate():
    get_corpus(); get_split(); get_tokenizer()
    res = evaluate()
    if res and res[0]:
        show_results(*res)
    return res

## Library L8 · Leak-free Test-data evaluation (same set as m2mTani-base)

The provided `Test-data/` English sentences, filtered by `00_build_leakfree_testset.ipynb` of the bundle: only verses
held out from training in every language and not near-copies of training text (109 test + 132 dev sentences; tier
`test` is the headline). English → language, beam 5, chrF++ / BLEU / TER with 95 % bootstrap intervals, compared with
m2mTani-base on exactly the same sentences.

In [ ]:
def bootstrap_ci(hyps, refs, n_boot=1000, seed=13):
    streams = [[r[k] if k < len(r) else None for r in refs] for k in range(max(map(len, refs)))]
    m = CHRF(word_order=2)
    stats = np.array(m._extract_corpus_statistics(hyps, streams))
    rng = np.random.default_rng(seed)
    vals = [m._compute_score_from_stats(list(stats[rng.integers(0, len(hyps), len(hyps))].sum(0))).score for _ in range(n_boot)]
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]


def evaluate_leakfree():
    if not os.path.isdir(FINAL_DIR):
        print(f'No fine-tuned model at {FINAL_DIR}: run T6 first.'); return None
    lf = json.load(open(cfg.leakfree_path, encoding='utf-8'))
    mdl = load_final()
    out, rows = {}, []
    for code, items in lf['languages'].items():
        for tier in ('test', 'dev'):
            sel = [r for r in items if r['tier'] == tier and r['refs']]
            if not sel:
                continue
            hyps = translate_batch(mdl, [r['source'] for r in sel], 'eng', code, beam=cfg.beam)
            refs = [r['refs'] for r in sel]
            streams = [[r[k] if k < len(r) else None for r in refs] for k in range(max(map(len, refs)))]
            out[f'eng>{code}|{tier}'] = {'n': len(sel), 'chrf++': CHRF(word_order=2).corpus_score(hyps, streams).score,
                                        'bleu': BLEU().corpus_score(hyps, streams).score, 'ter': TER().corpus_score(hyps, streams).score,
                                        'chrf++_ci95': bootstrap_ci(hyps, refs)}
            rows += [{'direction': f'eng>{code}', 'tier': tier, 'line': r['line'], 'verse': r['verse'], 'source': r['source'],
                      'hypothesis': h, 'reference': ' ||| '.join(r['refs'])} for r, h in zip(sel, hyps)]
    pd.DataFrame(rows).to_csv(os.path.join(RUN_DIR, 'leakfree_predictions.csv'), index=False)
    json.dump({'model': cfg.model_name, 'metrics': out}, open(os.path.join(RUN_DIR, 'leakfree_metrics.json'), 'w'), indent=1)
    base = json.load(open(cfg.base_leakfree_metrics_path))['metrics'] if os.path.exists(cfg.base_leakfree_metrics_path) else {}
    table = []
    for k, m in sorted(out.items()):
        b = base.get(k, {})
        ci = m['chrf++_ci95']
        table.append([k.replace('|', ' · '), m['n'], f"{b['chrf++']:.1f}" if b else '–', f"{m['chrf++']:.1f}",
                      f"{ci[0]:.1f}–{ci[1]:.1f}", f"{m['chrf++'] - b['chrf++']:+.1f}" if b else '–', f"{m['bleu']:.1f}"])
    print('### Leak-free Test-data: NLLB fine-tuned vs m2mTani-base (same sentences, English → language, beam 5)\n')
    print(md_table(['direction · tier', 'n', 'base chrF++', 'NLLB-ft chrF++', '95% CI', 'Δ', 'NLLB-ft BLEU'], table))
    return out


def translate(text, tgt='tgj', src='eng', beam=None):
    get_tokenizer()
    global _MODEL
    if '_MODEL' not in globals() or _MODEL is None:
        _MODEL = load_final()
    texts = [text] if isinstance(text, str) else list(text)
    res = translate_batch(_MODEL, [unicodedata.normalize('NFKC', t) for t in texts], src, tgt, beam=beam or cfg.beam)
    return res[0] if isinstance(text, str) else res


_MODEL = None

# Tasks

Each task cell loads or builds what it needs (after the library cells). `MTANI_STAGE` selects tasks in batch runs: `prepare` = T1–T5, `train` = T6, `evaluate` = T7–T9.

## T1 · Corpus

In [ ]:
if RUN('prepare'):
    get_corpus()
else:
    print('Skipped (MTANI_STAGE = {})'.format(STAGE))

## T2 · Split: the exact partition of m2mTani-base

In [ ]:
if RUN('prepare'):
    split_report()
else:
    print('Skipped (MTANI_STAGE = {})'.format(STAGE))

## T3 · Tokenizer and the new language codes

In [ ]:
if RUN('prepare'):
    tokenizer_report()
else:
    print('Skipped (MTANI_STAGE = {})'.format(STAGE))

## T4 · Training stream preview

In [ ]:
if RUN('prepare'):
    stream_preview()
else:
    print('Skipped (MTANI_STAGE = {})'.format(STAGE))

## T5 · Model

In [ ]:
if RUN('prepare'):
    model_summary()
else:
    print('Skipped (MTANI_STAGE = {})'.format(STAGE))

## T6 · Fine-tuning (resumes automatically)

In [ ]:
if RUN('train'):
    task_train()
else:
    print('Skipped (MTANI_STAGE = {})'.format(STAGE))

## T7 · Test-chapter evaluation + comparison with m2mTani-base

In [ ]:
if RUN('evaluate'):
    EVAL = task_evaluate()
else:
    print('Skipped (MTANI_STAGE = {})'.format(STAGE))

## T8 · Leak-free Test-data evaluation + comparison

In [ ]:
if RUN('evaluate'):
    get_corpus(); get_split(); LEAKFREE = evaluate_leakfree()
else:
    print('Skipped (MTANI_STAGE = {})'.format(STAGE))

## T9 · Translation demo

In [ ]:
if RUN('evaluate'):
    for tgt in ['tgj', 'njz', 'adi', 'adl', 'apt', 'mrg', 'hru', 'hin']:
        print(f'{tgt}:', translate('Thank you for helping me.', tgt))
else:
    print('Skipped (MTANI_STAGE = {})'.format(STAGE))